# **Day 8 - Calibration & SHAP (local run)**

This is the **local** counterpart of the Day 7 tuning work. It loads the tuned XGBoost from `artifacts/` (`.pkl` preferred, `.json` booster fallback), then performs README §7.4 (calibration + business threshold) and §7.5 (SHAP explainability). No Colab / Drive dependencies.

Inputs (all local):
- `data/processed/modelling/` : `X_val.parquet`, `X_test.parquet`, `y_val.parquet`, `y_test.parquet`, `day7_model_features.csv`, `day6_combined_importance.csv`, `val_demographics_lookup.parquet`
- `artifacts/` : `hmda_tuned_xgboost.pkl` (or `.json`)
- Tuned XGBoost, 78-feature canonical schema, `tract_minority_population_percent` flagged `is_race_proxy=1`.

### **Overview & decisions**
- **Model source:** `artifacts/hmda_tuned_xgboost.pkl` (reloadable XGBClassifier); falls back to `.json` booster if needed.
- **Raw vs calibrated:** `scale_pos_weight` distorts probabilities, so we calibrate (Platt/logistic) on `X_val` only and use the **calibrated** proba for the business threshold / risk tiers, but keep the **raw** model for SHAP (monotonic calibration preserves ranking and SHAP additivity).
- **SHAP:** `TreeExplainer` on a stratified 50k explanation sample drawn from `X_val` (aligned to `val_demographics_lookup.parquet` for the race-proxy step).
- **Protocol:** `X_test` is used only for the final confusion-matrix look; tuning/calibration use `X_val`.
- **Out of scope (Day 9/10):** subgroup fairness metrics, re-tuning, deployment.

In [1]:
import os, time, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import joblib, xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve,
                             precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
warnings.filterwarnings('ignore')

ROOT = Path('/Volumes/Mitul/Projects/home-mortage-approval-predictor')
M = ROOT / 'data' / 'processed' / 'modelling'
ART = ROOT / 'artifacts'
FIG = ROOT / 'figures' / 'day8'; FIG.mkdir(parents=True, exist_ok=True)
MARK = ROOT / 'markdown'; MARK.mkdir(exist_ok=True)
print('ROOT', ROOT)
print('xgboost', xgb.__version__, '| shap', shap.__version__)

ROOT /Volumes/Mitul/Projects/home-mortage-approval-predictor
xgboost 3.4.1 | shap 0.52.0


/Volumes/Mitul/Projects/home-mortage-approval-predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Step 0 - Pre-flight: column-order guard + model reload & metric reproduction**

Loads the canonical 78-feature schema from `day7_model_features.csv` and **asserts its column order exactly matches** `X_val`/`X_test`. A silent column shuffle would silently break every downstream prediction, so this guard fails loud rather than wrong. It then reloads the tuned XGBoost from `artifacts/hmda_tuned_xgboost.pkl` (falling back to the `.json` booster if needed) and **reproduces** the Day 7 hold-out metrics on `X_val`/`X_test` as proof we loaded the correct artifact. Nothing is trained in this step.

In [2]:
feat = pd.read_csv(M / 'day7_model_features.csv')
assert len(feat) == 78, f"expected 78 features, got {len(feat)}"
FEATURES = feat['feature'].tolist()
assert set(FEATURES).issubset(set(pd.read_parquet(M / 'X_val.parquet').columns))
print('loaded', len(FEATURES), 'model features | race-proxy flagged:', int(feat['is_race_proxy'].sum()))

Xval = pd.read_parquet(M / 'X_val.parquet').astype(np.float32)
yval = pd.read_parquet(M / 'y_val.parquet')['approved'].astype('int8').values
Xtest = pd.read_parquet(M / 'X_test.parquet').astype(np.float32)
ytest = pd.read_parquet(M / 'y_test.parquet')['approved'].astype('int8').values
Xv, Xt = Xval[FEATURES].values, Xtest[FEATURES].values

# Standing guard: array column order must match the CSV row order exactly
assert list(Xval[FEATURES].columns) == FEATURES, 'FEATURE column order mismatch vs day7_model_features.csv'
print('alignment guard passed: array order == CSV order')

# Load the tuned model
model = None
try:
    model = joblib.load(ART / 'hmda_tuned_xgboost.pkl')
    print('loaded model from .pkl ->', type(model).__name__)
except Exception as e:
    print('pkl load failed:', repr(e))
    bst = xgb.Booster(); bst.load_model(ART / 'hmda_tuned_xgboost.json'); model = bst
    print('loaded model from .json -> Booster')

def raw_proba(X):
    if isinstance(model, xgb.Booster):
        return model.predict(xgb.DMatrix(X))
    return model.predict_proba(X)[:, 1]

p_val = raw_proba(Xv); p_test = raw_proba(Xt)
print('REPRODUCED  val  ROC-AUC=%.4f PR-AUC=%.4f' % (roc_auc_score(yval, p_val), average_precision_score(yval, p_val)))
print('REPRODUCED test  ROC-AUC=%.4f PR-AUC=%.4f' % (roc_auc_score(ytest, p_test), average_precision_score(ytest, p_test)))

loaded 78 model features | race-proxy flagged: 1
alignment guard passed: array order == CSV order
loaded model from .pkl -> XGBClassifier
REPRODUCED  val  ROC-AUC=0.8930 PR-AUC=0.9531
REPRODUCED test  ROC-AUC=0.8933 PR-AUC=0.9532


## **Step 1 - Calibration check (Platt scaling on X_val only)**

`scale_pos_weight` intentionally distorts the raw scores toward the minority (deny) class, so the model's raw probabilities are no longer well-calibrated. We fit a **Platt (logistic) scaler** on `X_val` *only* (never `X_test`) to map raw scores to true P(approve), and plot reliability diagrams before/after. Because logistic calibration is **monotonic**, it preserves score ranking and SHAP additivity — so the raw model is still used for SHAP in Steps 3-6, while the calibrated scores drive the business threshold in Step 2.

In [3]:
# Raw reliability diagram on X_val
fraction_of_pos, mean_pred = calibration_curve(yval, p_val, n_bins=10, strategy='quantile')
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.plot(mean_pred, fraction_of_pos, 'o-', label='raw model')
plt.xlabel('mean predicted P(approve)'); plt.ylabel('observed approval rate')
plt.title('Calibration (X_val) - raw'); plt.legend(); plt.savefig(FIG / 'calibration_raw.png', dpi=120); plt.close()

# Platt scaling (logistic) fit on X_val only -> monotonic, preserves ranking & SHAP additivity
cal = LogisticRegression(); cal.fit(p_val.reshape(-1, 1), yval)
pc_val = cal.predict_proba(p_val.reshape(-1, 1))[:, 1]
pc_test = cal.predict_proba(p_test.reshape(-1, 1))[:, 1]

frac2, mean2 = calibration_curve(yval, pc_val, n_bins=10, strategy='quantile')
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.plot(mean2, frac2, 'o-', label='calibrated (Platt)')
plt.xlabel('mean predicted P(approve)'); plt.ylabel('observed approval rate')
plt.title('Calibration (X_val) - after Platt'); plt.legend(); plt.savefig(FIG / 'calibration_calibrated.png', dpi=120); plt.close()

print('Calibration decision: applied Platt scaling on X_val; RAW model retained for SHAP (Steps 3-6).')
print('Calibrated val  ROC-AUC=%.4f PR-AUC=%.4f (monotonic -> ranking unchanged)' % (roc_auc_score(yval, pc_val), average_precision_score(yval, pc_val)))

Calibration decision: applied Platt scaling on X_val; RAW model retained for SHAP (Steps 3-6).
Calibrated val  ROC-AUC=0.8930 PR-AUC=0.9531 (monotonic -> ranking unchanged)


## **Step 2 - Business-relevant operating threshold (precision-first)**

For a lender, a **false positive** (approving an applicant who later defaults) loses the full principal + interest, whereas a **false negative** (denying a creditworthy applicant) loses only forgone interest. The dominant financial risk is therefore **over-approval**, so the operating threshold is chosen to favor **precision** (few false approvals), not F1/recall.

We select the threshold via an **Fbeta score with BETA=0.3** (precision-weighted) on the calibration set; the F1-optimal point is shown only for comparison. Raising the threshold to protect precision **increases the false-negative rate** (more creditworthy applicants denied), which becomes the fair-lending-sensitive metric -- Day 9 must report FNR by subgroup.

A **confusion-matrix figure** is written at this threshold (`figures/day8/confusion_matrix.png`). Read it as: the **top-right (FP)** cell is the costly *over-approval* the precision-first threshold is designed to suppress, while the **bottom-left (FN)** cell is the fair-lending-sensitive false denial that Day 9 must examine by subgroup.

In [4]:
BETA = 0.3   # precision-weighted: FP (approve-a-defaulter) costs principal+interest >> FN (deny creditworthy) costs forgone interest
prec, rec, thr = precision_recall_curve(yval, pc_val)
# Fbeta (precision-weighted) threshold -- CHOSEN operating point
fb = (1 + BETA**2) * prec[:-1] * rec[:-1] / (BETA**2 * prec[:-1] + rec[:-1] + 1e-12)
fb_thr = thr[np.argmax(fb)]
# F1-optimal shown only for comparison
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
f1_thr = thr[np.argmax(f1s)]
THRESH = fb_thr  # chosen: precision-weighted (lender-cost rationale)
print('Fbeta(B=%.1f) threshold   = %.3f  [CHOSEN]' % (BETA, fb_thr))
print('F1-optimal threshold      = %.3f  (comparison only)' % f1_thr)

def report_at(t, y, p, name):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    tpr = tp / (tp + fn); fpr = fp / (fp + tn); fnr = fn / (fn + tp)
    out = dict(tpr=tpr, fpr=fpr, fnr=fnr, precision=precision_score(y, pred),
               recall=recall_score(y, pred), f1=f1_score(y, pred), cm=confusion_matrix(y, pred))
    print('[%s @%.3f] TPR=%.4f FPR=%.4f FNR=%.4f P=%.4f R=%.4f F1=%.4f' % (name, t, tpr, fpr, fnr, out['precision'], out['recall'], out['f1']))
    return out

res_val = report_at(THRESH, yval, pc_val, 'X_val')
res_test = report_at(THRESH, ytest, pc_test, 'X_test')
print('FPR = share of actually-denied applicants wrongly predicted approve = costly over-approval (principal + interest at risk). Primary financial risk under lender-cost view.')
print('FNR = share of actually-approved applicants wrongly predicted denied = fair-lending-sensitive error (Day 9 reports by subgroup).')

Fbeta(B=0.3) threshold   = 0.858  [CHOSEN]
F1-optimal threshold      = 0.345  (comparison only)
[X_val @0.858] TPR=0.7347 FPR=0.1456 FNR=0.2653 P=0.9369 R=0.7347 F1=0.8236
[X_test @0.858] TPR=0.7357 FPR=0.1458 FNR=0.2643 P=0.9370 R=0.7357 F1=0.8242
FPR = share of actually-denied applicants wrongly predicted approve = costly over-approval (principal + interest at risk). Primary financial risk under lender-cost view.
FNR = share of actually-approved applicants wrongly predicted denied = fair-lending-sensitive error (Day 9 reports by subgroup).


In [5]:
# Confusion-matrix visualization at the chosen precision-weighted threshold.
# sklearn default label order [0,1] = [Denied, Approved]:
#   TN  FP   <- top-right (FP) = costly OVER-approval (principal + interest at risk)
#   FN  TP   <- bottom-left (FN) = fair-lending-sensitive false denial
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (cm, name) in zip(axes, [(res_val['cm'], 'X_val'), (res_test['cm'], 'X_test')]):
    disp = ConfusionMatrixDisplay(cm, display_labels=['Denied', 'Approved'])
    disp.plot(ax=ax, cmap='Blues', values_format=',d')
    ax.set_title('Confusion matrix @ THRESH=%.3f (%s)' % (THRESH, name))
plt.tight_layout(); plt.savefig(FIG / 'confusion_matrix.png', dpi=120); plt.close()
print('wrote', FIG / 'confusion_matrix.png')

wrote /Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/day8/confusion_matrix.png


## **Step 3 - SHAP setup at scale**

Draws a stratified 50k-row explanation sample from `X_val` and aligns it **positionally** with `val_demographics_lookup.parquet`, so the demographic subgroup analysis in Step 6 uses the exact same rows. It fits a `TreeExplainer` on the raw model and computes SHAP values for every feature of every sampled application. `sv[i, j]` is the contribution of feature `j` to the log-odds of approval for application `i`; `expected_value` is the dataset baseline log-odds.

In [6]:
N_EXPLAIN = 50_000
rng = np.random.RandomState(42)
pos = rng.choice(len(Xv), size=N_EXPLAIN, replace=False)
X_explain = Xv[pos]
demo = pd.read_parquet(M / 'val_demographics_lookup.parquet').iloc[pos].reset_index(drop=True)
assert len(demo) == N_EXPLAIN, 'demographics alignment failed'
print('explanation sample:', X_explain.shape, '| demographics aligned:', len(demo) == N_EXPLAIN)

t0 = time.time()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_explain)
if isinstance(sv, list):
    sv = sv[1]
ev = explainer.expected_value
if isinstance(ev, (list, np.ndarray)):
    ev = float(np.asarray(ev)[1]) if np.asarray(ev).size > 1 else float(np.asarray(ev).item())
print('SHAP done in %.1fs | sv shape %s | expected_value %.5f' % (time.time() - t0, sv.shape, ev))

explanation sample: (50000, 78) | demographics aligned: True
SHAP done in 1164.6s | sv shape (50000, 78) | expected_value -0.00079


## **Step 4 - Global interpretability**

Aggregates the 50k SHAP matrix into two views: a **beeswarm** (per-feature distribution of impact, colored by feature value) and a **bar chart** (mean |SHAP| per feature = global importance). We also build a combined cross-method importance table (`day8_importance.csv`) joining the SHAP rank with the Day 6 gain ranks (XGBoost / LightGBM / CatBoost / RF) and a fresh logistic-regression coefficient rank, so feature importance can be compared across five independent methods.

In [7]:
plt.figure(); shap.summary_plot(sv, X_explain, feature_names=FEATURES, show=False, max_display=25)
plt.tight_layout(); plt.savefig(FIG / 'shap_beeswarm.png', dpi=120); plt.close()
plt.figure(); shap.summary_plot(sv, X_explain, plot_type='bar', feature_names=FEATURES, show=False, max_display=25)
plt.tight_layout(); plt.savefig(FIG / 'shap_bar.png', dpi=120); plt.close()

# Combined cross-method importance table
shap_abs = np.abs(sv).mean(0)
imp = pd.DataFrame({'feature': FEATURES, 'shap_mean_abs': shap_abs})
imp['shap_rank'] = imp['shap_mean_abs'].rank(ascending=False).astype(int)
prior = pd.read_csv(M / 'day6_combined_importance.csv')
imp = imp.merge(prior, on='feature', how='left')
# LR coefficient rank (quick, on explanation sample)
lr = LogisticRegression(max_iter=2000, class_weight='balanced')
lr.fit(X_explain, yval[pos])
imp['lr_coef_abs'] = np.abs(lr.coef_[0])
imp['lr_coef_rank'] = imp['lr_coef_abs'].rank(ascending=False).astype(int)
imp = imp.sort_values('shap_mean_abs', ascending=False).reset_index(drop=True)
imp.to_csv(M / 'day8_importance.csv', index=False)
print('top 15 by SHAP mean|value|:')
print(imp[['feature', 'shap_mean_abs', 'shap_rank', 'XGB_gain_rank', 'LGBM_gain_rank', 'RF_rank', 'lr_coef_rank']].head(15).to_string(index=False))

top 15 by SHAP mean|value|:
                                                         feature  shap_mean_abs  shap_rank  XGB_gain_rank  LGBM_gain_rank  RF_rank  lr_coef_rank
                                                  loan_purpose_1       0.504645          1            3.0             1.0       12            18
                                    debt_to_income_ratio_missing       0.377553          2            7.0             5.0       14            16
                                             loan_to_value_ratio       0.342529          3           33.0             4.0        3             9
                                                          income       0.267513          4           24.0             3.0        1             7
                                                  property_value       0.262334          5           44.0             7.0        6             3
                                            loan_to_income_ratio       0.167078          6           3

## **Step 5 - Local (per-application) interpretability**

Defines `explain_application(i)`, which turns a single SHAP vector into a human-readable explanation: calibrated P(approve), denial risk, the README risk tier, and the top contributing factors (positive = pushes toward approval, negative = toward denial). It is run on three illustrative cases — the highest-P(approve), the lowest-P(approve), and the borderline case nearest the operating threshold — each with a SHAP **waterfall** plot showing how the baseline log-odds is pushed step-by-step to the final prediction.

In [8]:
def risk_tier(p):
    if p < 0.20: return 'Low (high approval prob)'
    if p < 0.50: return 'Moderate'
    if p < 0.75: return 'High'
    return 'Very high (low approval prob)'

def explain_application(i):
    raw = p_val[pos][i]
    p = float(cal.predict_proba(np.array([[raw]]))[:, 1][0])
    denial = 1 - p
    contrib = sv[i]
    order = np.argsort(-np.abs(contrib))
    print('--- Application %d ---' % i)
    print('Predicted P(approve) = %.3f | denial risk = %.3f | tier: %s' % (p, denial, risk_tier(denial)))
    print('Actual approved = %d' % yval[pos][i])
    print('Top factors (SHAP, log-odds space):')
    for j in order[:8]:
        f = FEATURES[j]; v = contrib[j]
        print('  %-45s %+.4f %s' % (f, v, '-> approval' if v > 0 else '-> denial'))
    return p, denial, contrib

pc_pos = cal.predict_proba(p_val[pos].reshape(-1, 1))[:, 1]
hi_app = int(np.argmax(pc_pos))
hi_den = int(np.argmin(pc_pos))
border = int(np.argmin(np.abs(pc_pos - THRESH)))
for i in [hi_app, hi_den, border]:
    explain_application(i)
    exp_i = shap.Explanation(values=sv[i], base_values=ev, data=X_explain[i], feature_names=FEATURES)
    plt.figure(); shap.plots.waterfall(exp_i, max_display=12, show=False)
    plt.tight_layout(); plt.savefig(FIG / ('waterfall_%d.png' % i), dpi=120); plt.close()
    print()
print('wrote waterfall plots for 3 illustrative cases')

--- Application 7184 ---
Predicted P(approve) = 0.983 | denial risk = 0.017 | tier: Low (high approval prob)
Actual approved = 1
Top factors (SHAP, log-odds space):
  preapproval_1                                 +4.1048 -> approval
  preapproval_2                                 +1.7372 -> approval
  loan_term                                     +0.9280 -> approval
  loan_purpose_1                                +0.7714 -> approval
  intro_rate_period                             +0.1792 -> approval
  loan_to_value_ratio                           +0.1756 -> approval
  property_value                                +0.1558 -> approval
  debt_to_income_ratio_missing                  -0.1449 -> denial

--- Application 18263 ---
Predicted P(approve) = 0.101 | denial risk = 0.899 | tier: Very high (low approval prob)
Actual approved = 0
Top factors (SHAP, log-odds space):
  property_value                                -1.5896 -> denial
  derived_dwelling_category_Single Family (1-4 Units):M

## **Step 6 - Race-proxy SHAP check (by derived_race)**

`tract_minority_population_percent` is flagged `is_race_proxy=1` because it is geographically correlated with race and was deliberately kept in the model. Here we take its SHAP column and group the mean (and mean-|value|) by `derived_race` to see whether the proxy feature pushes different demographic groups in different directions. This is a **descriptive proxy signal only** — it does *not* by itself prove disparate error rates; that causal/error-rate question is the job of Day 9.

In [9]:
j = FEATURES.index('tract_minority_population_percent')
rank_row = imp.loc[imp.feature == FEATURES[j], 'shap_rank']
print('tract_minority_population_percent SHAP rank:', int(rank_row.iloc[0]) if len(rank_row) else 'n/a')
proxy_sv = sv[:, j]
demo['proxy_sv'] = proxy_sv
grp = demo.groupby('derived_race')['proxy_sv'].agg(mean='mean', abs_mean=lambda s: s.abs().mean(), count='count').sort_values('mean')
print(grp)
plt.figure(figsize=(7, 4))
grp['mean'].plot(kind='barh'); plt.xlabel('mean SHAP (tract_minority_population_percent)')
plt.title('Proxy-feature SHAP by derived_race'); plt.tight_layout(); plt.savefig(FIG / 'proxy_shap_by_race.png', dpi=120); plt.close()
print('Positive mean => feature pushes this group toward approval; negative => toward denial.')

tract_minority_population_percent SHAP rank: 7
                                               mean  abs_mean  count
derived_race                                                        
American Indian or Alaska Native           0.009835  0.143202    511
White                                      0.010771  0.146905  32346
Black or African American                  0.011555  0.146868   5210
Race Not Available                         0.012741  0.147580   8633
Asian                                      0.016763  0.147353   1845
Native Hawaiian or Other Pacific Islander  0.018807  0.144174    149
2 or more minority races                   0.019478  0.151259    162
Joint                                      0.020836  0.141964   1143
Free Form Text Only                        0.241834  0.241834      1
Positive mean => feature pushes this group toward approval; negative => toward denial.


## **Step 7 - Wrap-up summary**

Collects the key numbers from Steps 0-6 into `markdown/day8_calibration_shap_summary.md`: the reproduced metrics, the chosen precision-weighted threshold and its confusion-matrix breakdown, the top SHAP features, and the race-proxy finding. This is the **hand-off artifact** for Day 9 (fairness metrics) and Day 10 (deployment).

In [10]:
lines = [
    "# Day 8 - Calibration & SHAP Summary", "",
    "- Model: Day 7 tuned XGBoost (20-trial Optuna). Reproduced val ROC-AUC=%.4f, test ROC-AUC=%.4f." % (roc_auc_score(yval, p_val), roc_auc_score(ytest, p_test)),
    "- Calibration: applied Platt (logistic) scaling on X_val only; RAW model retained for SHAP. Business output read as calibrated P(approve).",
    "- Operating threshold (X_val Fbeta=0.3 precision-weighted) = %.3f." % THRESH,
    "  X_val : TPR=%.4f FPR=%.4f FNR=%.4f P=%.4f R=%.4f F1=%.4f" % (res_val['tpr'], res_val['fpr'], res_val['fnr'], res_val['precision'], res_val['recall'], res_val['f1']),
    "  X_test: TPR=%.4f FPR=%.4f FNR=%.4f P=%.4f R=%.4f F1=%.4f" % (res_test['tpr'], res_test['fpr'], res_test['fnr'], res_test['precision'], res_test['recall'], res_test['f1']),
    "- SHAP: TreeExplainer on raw model, stratified sample n=%d (X_val)." % N_EXPLAIN,
    "- Top features by |SHAP|: " + ", ".join(imp.head(5)['feature'].tolist()),
    "- Race-proxy (tract_minority_population_percent): mean SHAP by derived_race computed (see proxy_shap_by_race.png).",
    "- Confusion matrices at THRESH: see figures/day8/confusion_matrix.png.",
    "- Open Q for Day 9: does proxy SHAP translate into subgroup error/calibration disparity? Use 0.5 or business threshold? Other geo-adjacent proxies?",
]
summary = "\n".join(lines)
print(summary)
open(MARK / 'day8_calibration_shap_summary.md', 'w').write(summary)
print('wrote', MARK / 'day8_calibration_shap_summary.md')

# Day 8 - Calibration & SHAP Summary

- Model: Day 7 tuned XGBoost (20-trial Optuna). Reproduced val ROC-AUC=0.8930, test ROC-AUC=0.8933.
- Calibration: applied Platt (logistic) scaling on X_val only; RAW model retained for SHAP. Business output read as calibrated P(approve).
- Operating threshold (X_val Fbeta=0.3 precision-weighted) = 0.858.
  X_val : TPR=0.7347 FPR=0.1456 FNR=0.2653 P=0.9369 R=0.7347 F1=0.8236
  X_test: TPR=0.7357 FPR=0.1458 FNR=0.2643 P=0.9370 R=0.7357 F1=0.8242
- SHAP: TreeExplainer on raw model, stratified sample n=50000 (X_val).
- Top features by |SHAP|: loan_purpose_1, debt_to_income_ratio_missing, loan_to_value_ratio, income, property_value
- Race-proxy (tract_minority_population_percent): mean SHAP by derived_race computed (see proxy_shap_by_race.png).
- Confusion matrices at THRESH: see figures/day8/confusion_matrix.png.
- Open Q for Day 9: does proxy SHAP translate into subgroup error/calibration disparity? Use 0.5 or business threshold? Other geo-adjacent p